# Fabric Defect Inspector: speed-up

The inspector runs at about 1.6 frames per second. The target is 5. This notebook measures where the time goes, tries three changes, and checks that accuracy does not drop.

1. **Half precision**: both models run in 16-bit floats on the GPU.
2. **Smaller PatchCore memory bank**: keep 25%, 10% or 5% of the 55,440 normal patches, picked so they still cover the full variety of clean fabric (greedy coreset).
3. **PatchCore on every 3rd or 5th frame**: YOLO still runs on every frame.

For every option it reports speed, defects found on the 140 test photos, and false alarms. The pixel threshold is set again on validation photos for each option, because a smaller bank changes the scores. It then picks the fastest option that keeps accuracy and re-runs the simulated roll to check the count.

**Needs in Google Drive (already there)**
- `My Drive/textile_yolo.zip`
- `My Drive/fabric_inspector_yolo/best.pt`
- `My Drive/fabric_inspector_patchcore/patchcore_textile.pt`

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Takes roughly 20 to 30 minutes.

## 1. Setup

In [ ]:
!pip install -q ultralytics
import time
import torch, torchvision, numpy as np, cv2, pathlib, shutil, zipfile, yaml, random, collections, os
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
dev = 'cuda'
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive')
ZIP_PATH = DRIVE / 'textile_yolo.zip'
YOLO_PT = DRIVE / 'fabric_inspector_yolo' / 'best.pt'
PC_PT = DRIVE / 'fabric_inspector_patchcore' / 'patchcore_textile.pt'
for p in (ZIP_PATH, YOLO_PT, PC_PT):
    assert p.exists(), f"Not found in Drive: {p}"
DATA_ROOT = pathlib.Path('/content/data')
OUT = pathlib.Path('/content/speed_out'); OUT.mkdir(exist_ok=True)

## 2. Load the test and validation photos

In [ ]:
random.seed(0); np.random.seed(0)
if ZIP_PATH is not None:
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = DATA_ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

DS = list(DATA_ROOT.rglob('data.yaml'))[0].parent
NAMES = yaml.safe_load(open(DS / 'data.yaml'))['names']
NAMES = list(NAMES.values()) if isinstance(NAMES, dict) else list(NAMES)

def read_labels(img_path, W, H):
    """Ground-truth boxes for one image: [(class name, x1, y1, x2, y2), ...] in pixels."""
    lp = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    out = []
    if lp.exists():
        for v in (l.split() for l in open(lp) if l.strip()):
            c, cx, cy, w, h = int(v[0]), *map(float, v[1:5])
            out.append((NAMES[c], (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
    return out

test_imgs = sorted((DS / 'test' / 'images').glob('*'))
valid_imgs = sorted((DS / 'valid' / 'images').glob('*'))
print(f"classes {NAMES}   test photos {len(test_imgs)}   validation photos {len(valid_imgs)}")

## 3. Load both models
Two copies of YOLO are loaded, one normal and one half precision, so each option is timed cleanly.

In [ ]:
from ultralytics import YOLO
CONF = 0.25
yolo_full, yolo_half = YOLO(str(YOLO_PT)), YOLO(str(YOLO_PT))
torch.backends.cudnn.benchmark = True

ck = torch.load(PC_PT, map_location='cpu')
bank_full = ck['bank'].float().to(dev)
CROP_THRESHOLD = float(ck['threshold'])
net = torchvision.models.wide_resnet50_2(weights=torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1).to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
print(f"PatchCore bank {tuple(bank_full.shape)}")

def yolo_detect(bgr, half=False):
    """[(class name, confidence, x1, y1, x2, y2), ...] in pixels."""
    m = yolo_half if half else yolo_full
    r = m.predict(bgr, imgsz=max(bgr.shape[:2]), conf=CONF, half=half, verbose=False)[0]
    return [(m.names[int(c)], float(p), *map(float, b))
            for b, p, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())]

@torch.no_grad()
def features(bgr, half=False):
    """One feature vector per 8x8 px patch. Returns (patches x channels, rows, cols)."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None].to(dev)
    x = (x - MEAN) / STD
    with torch.autocast('cuda', dtype=torch.float16, enabled=half):
        x = net.maxpool(net.relu(net.bn1(net.conv1(x))))
        l2 = net.layer2(net.layer1(x)); l3 = net.layer3(l2)
        f2 = F.avg_pool2d(l2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(l3, 3, 1, 1), size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1)[0]
    f = f.float()
    C, h, w = f.shape
    return f.permute(1, 2, 0).reshape(-1, C), h, w

class Bank:
    """A memory bank with a fast nearest-neighbour distance."""
    def __init__(self, b):
        self.b = b.contiguous(); self.b2 = (self.b * self.b).sum(1)
    def __len__(self):
        return self.b.shape[0]
    @torch.no_grad()
    def dist(self, q):
        out = []
        for i in range(0, q.shape[0], 4096):
            qi = q[i:i + 4096]
            d2 = (qi * qi).sum(1, keepdim=True) + self.b2[None] - 2 * (qi @ self.b.T)
            out.append(d2.min(1).values)
        return torch.cat(out).clamp_min(0).sqrt()

@torch.no_grad()
def anomaly_map(bgr, v):
    """Distance of every patch to its nearest normal patch, for one option v. Shape (H/8, W/8)."""
    q, h, w = features(bgr, v['half'])
    if v['original']:      # exactly what the fusion notebook did
        d = torch.cat([torch.cdist(q[i:i + 4096], v['bank'].b).min(1).values for i in range(0, q.shape[0], 4096)])
    else:
        d = v['bank'].dist(q)
    return d.view(h, w).cpu().numpy()

## 4. Fusion logic
Same as the fusion notebook.

In [ ]:
CELL = 8            # one anomaly-map cell covers 8 x 8 px
MIN_CELLS = 4       # smallest region that counts
COLORS = {'Hole': (60, 60, 240), 'Knot': (0, 145, 255), 'Line': (255, 200, 0), 'Stain': (216, 50, 216), 'unknown': (0, 220, 255)}  # BGR

def smooth(amap):
    return cv2.GaussianBlur(amap.astype(np.float32), (0, 0), 1.0)

def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def overlaps(a, b):
    return min(a[2], b[2]) > max(a[0], b[0]) and min(a[3], b[3]) > max(a[1], b[1])

def centre_in(a, b):
    cx, cy = (a[0] + a[2]) / 2, (a[1] + a[3]) / 2
    return b[0] <= cx <= b[2] and b[1] <= cy <= b[3]

def same_defect(a, b):
    return iou(a, b) >= 0.2 or centre_in(a, b) or centre_in(b, a)

def regions(amap_s, thr):
    """Anomalous regions in a smoothed map: [(score, x1, y1, x2, y2), ...] in pixels, plus the cell mask."""
    mask = (amap_s > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if a >= MIN_CELLS:
            out.append((float(amap_s[lab == i].max()), x * CELL, y * CELL, (x + w) * CELL, (y + h) * CELL))
    return out, mask

def inspect(bgr, thr, ignore_cols=None):
    """Fused result for one image or frame."""
    known = yolo_detect(bgr)
    amap_s = smooth(anomaly_map(bgr))
    if ignore_cols is not None and len(ignore_cols):
        amap_s[:, ignore_cols] = 0
    regs, mask = regions(amap_s, thr)
    unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
    return known, unknown, amap_s, mask

## 5. Build the smaller memory banks
Greedy coreset: start from one patch, then keep adding the patch that is furthest from everything picked so far. The first 25% picked this way already contain the first 10%, which contain the first 5%.

In [ ]:
@torch.no_grad()
def greedy_order(b, k, dim=128, seed=0):
    g = torch.Generator().manual_seed(seed)
    P = b @ (torch.randn(b.shape[1], dim, generator=g).to(b.device) / dim ** 0.5)     # random projection, only for picking
    idx = [0]
    min_d = torch.cdist(P, P[0:1]).squeeze(1)
    for _ in range(k - 1):
        i = int(min_d.argmax()); idx.append(i)
        min_d = torch.minimum(min_d, torch.cdist(P, P[i:i + 1]).squeeze(1))
    return torch.tensor(idx, device=b.device)

t0 = time.time()
N = bank_full.shape[0]
order = greedy_order(bank_full, int(N * 0.25))
print(f"picked {len(order):,} of {N:,} patches in {time.time() - t0:.0f} s")

full = Bank(bank_full)
VARIANTS = [
    {'name': 'original (as in the fusion notebook)', 'bank': full, 'half': False, 'original': True, 'fraction': 1.0},
    {'name': 'half precision, full bank', 'bank': full, 'half': True, 'original': False, 'fraction': 1.0},
]
for frac in (0.25, 0.10, 0.05):
    VARIANTS.append({'name': f'half precision, {int(frac * 100)}% bank', 'bank': Bank(bank_full[order[:int(N * frac)]]),
                     'half': True, 'original': False, 'fraction': frac})

## 6. Build the simulated roll
The same roll as the fusion notebook: 40 test photos, same random seed. Its frames are used for timing here and for the count check at the end.

In [ ]:
N_IMAGES, VIEW_W, STEP = 40, 960, 32
M_PER_IMAGE = 0.15
MIN_HITS = 2

random.seed(3)
roll_imgs = random.sample(test_imgs, min(N_IMAGES, len(test_imgs)))
tiles, gt_roll, seams, x0 = [], [], [], 0
for p in roll_imgs:
    im = cv2.imread(str(p)); H, W = im.shape[:2]
    if H != 640:
        im = cv2.resize(im, (int(W * 640 / H), 640)); H, W = im.shape[:2]
    for cls, x1, y1, x2, y2 in read_labels(p, W, H):
        gt_roll.append((cls, x0 + x1, y1, x0 + x2, y2))
    tiles.append(im); x0 += W; seams.append(x0)
roll = np.concatenate(tiles, axis=1)
ROLL_LEN = roll.shape[1]; PX_PER_M = (ROLL_LEN / len(tiles)) / M_PER_IMAGE
OFFSETS = list(range(0, ROLL_LEN - VIEW_W + 1, STEP))
print(f"roll: {len(tiles)} photos, {ROLL_LEN} px = {ROLL_LEN / PX_PER_M:.2f} m, {len(gt_roll)} labelled defects, {len(OFFSETS)} frames")

## 7. Speed and accuracy of every option
For each option:

- the pixel threshold is set again on validation photos (99.5th percentile of clean patch scores, as before);
- accuracy is measured on the 140 test photos, the same way as section 6 of the fusion notebook;
- speed is measured on 40 roll frames of 960 x 640 px. Only the two models are timed. Drawing and saving the video are not included, so the "original" speed here can be a little higher than the 1.6 fps seen before.

In [ ]:
PERCENTILE = 99.5

def calibrate(v):
    vals = []
    for p in valid_imgs[:150]:
        bgr = cv2.imread(str(p)); H, W = bgr.shape[:2]
        a = smooth(anomaly_map(bgr, v))
        clean = np.ones(a.shape, bool)
        for _, x1, y1, x2, y2 in read_labels(p, W, H):
            clean[max(int(y1 // CELL) - 2, 0):int(y2 // CELL) + 3, max(int(x1 // CELL) - 2, 0):int(x2 // CELL) + 3] = False
        vals.append(a[clean])
    return float(np.percentile(np.concatenate(vals), PERCENTILE))

yolo_cache = {h: {p: yolo_detect(cv2.imread(str(p)), h) for p in test_imgs} for h in (False, True)}

def evaluate(v, thr):
    n = hit_y = hit_p = hit_e = fa = 0
    for p in test_imgs:
        bgr = cv2.imread(str(p)); H, W = bgr.shape[:2]
        gts = read_labels(p, W, H)
        known = yolo_cache[v['half']][p]
        regs, mask = regions(smooth(anomaly_map(bgr, v)), thr)
        unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
        for cls, x1, y1, x2, y2 in gts:
            g = (x1, y1, x2, y2)
            y = any(same_defect(g, k[2:]) for k in known)
            cy1, cy2 = max(int(y1 // CELL) - 1, 0), int(y2 // CELL) + 2
            cx1, cx2 = max(int(x1 // CELL) - 1, 0), int(x2 // CELL) + 2
            pc = bool(mask[cy1:cy2, cx1:cx2].any())
            n += 1; hit_y += y; hit_p += pc; hit_e += (y or pc)
        gt_boxes = [g[1:] for g in gts]
        fa += sum(not any(same_defect(k[2:], g) for g in gt_boxes) for k in known)
        fa += sum(not any(overlaps(u[1:], g) for g in gt_boxes) for u in unknown)
    return {'found by YOLO': hit_y / n, 'found by PatchCore': hit_p / n, 'found by either': hit_e / n,
            'false alarms per photo': fa / len(test_imgs)}

time_frames = [np.ascontiguousarray(roll[:, o:o + VIEW_W]) for o in OFFSETS[::max(len(OFFSETS) // 40, 1)][:40]]

def time_models(v, thr):
    for f in time_frames[:3]:                                   # warm up
        yolo_detect(f, v['half']); regions(smooth(anomaly_map(f, v)), thr)
    torch.cuda.synchronize(); t = time.perf_counter()
    for f in time_frames:
        yolo_detect(f, v['half'])
    torch.cuda.synchronize(); ty = (time.perf_counter() - t) / len(time_frames)
    t = time.perf_counter()
    for f in time_frames:
        regions(smooth(anomaly_map(f, v)), thr)
    torch.cuda.synchronize(); tp = (time.perf_counter() - t) / len(time_frames)
    return ty, tp

rows = []
for v in VARIANTS:
    v['thr'] = calibrate(v)
    acc = evaluate(v, v['thr'])
    ty, tp = time_models(v, v['thr'])
    v.update(acc); v['ty'], v['tp'] = ty, tp
    rows.append({'option': v['name'], 'bank patches': len(v['bank']), 'pixel threshold': round(v['thr'], 2),
                 **{k: round(x, 3) for k, x in acc.items()},
                 'YOLO ms': round(ty * 1000, 1), 'PatchCore ms': round(tp * 1000, 1),
                 'fps, PatchCore every frame': round(1 / (ty + tp), 1),
                 'fps, PatchCore every 3rd frame': round(1 / (ty + tp / 3), 1),
                 'fps, PatchCore every 5th frame': round(1 / (ty + tp / 5), 1)})
    print(rows[-1])
speed_table = pd.DataFrame(rows)
speed_table.to_csv(OUT / 'speed_table.csv', index=False)
speed_table.T

## 8. Pick the fastest option that keeps accuracy
An option is accepted if, compared with the original, defects found by either model drop by no more than 1 point, defects found by PatchCore drop by no more than 3 points, and false alarms rise by no more than 0.03 per photo. The smallest accepted bank wins.

In [ ]:
base = VARIANTS[0]
def ok(v):
    return (v['found by either'] >= base['found by either'] - 0.01 and
            v['found by PatchCore'] >= base['found by PatchCore'] - 0.03 and
            v['false alarms per photo'] <= base['false alarms per photo'] + 0.03)
accepted = [v for v in VARIANTS[1:] if ok(v)]
if accepted:
    best = min(accepted, key=lambda v: (len(v['bank']), v['tp']))
    print("chosen:", best['name'])
else:
    best = base
    print("No faster option kept accuracy within the limits. Keeping the original. Look at the table above before deciding.")

## 9. Check the count on the simulated roll
The whole roll is run three times: the original, the chosen option with PatchCore on every frame, and the chosen option with PatchCore on every 3rd frame. The original should give the same count as the fusion notebook (53 of 67). The speed here is for the two models plus the counting, over all frames, without drawing or saving video.

In [ ]:
def same_track(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    if inter <= 0:
        return False
    area_a, area_b = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter) >= 0.2 or inter / min(area_a, area_b) >= 0.5

def run_roll(v, pc_every=1):
    tracks = []
    def add(kind, cls, conf, box):
        for t in tracks:
            if same_track(box, t['box']) or (kind != t['kind'] and overlaps(box, t['box'])):
                t['hits'] += 1
                if kind == 'known':
                    if t['kind'] == 'unknown' or conf > t['best']:
                        t['box'], t['best'] = list(box), conf
                    t['kind'] = 'known'; t['votes'][cls] = t['votes'].get(cls, 0) + conf
                return
        tracks.append({'box': list(box), 'best': conf if kind == 'known' else 0.0,
                       'votes': {cls: conf} if kind == 'known' else {}, 'hits': 1, 'kind': kind})
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for n, off in enumerate(OFFSETS):
        frame = np.ascontiguousarray(roll[:, off:off + VIEW_W])
        seam_x = [s - off for s in seams if 0 < s - off < VIEW_W]
        known = yolo_detect(frame, v['half'])
        known = [k for k in known if not (k[4] - k[2] <= 64 and any(k[2] - 8 < sx < k[4] + 8 for sx in seam_x))]
        unknown = []
        if n % pc_every == 0:
            ignore = sorted({c for s in seam_x for c in range(max(s // CELL - 5, 0), min(s // CELL + 5, VIEW_W // CELL))})
            amap_s = smooth(anomaly_map(frame, v))
            if ignore:
                amap_s[:, ignore] = 0
            regs, _ = regions(amap_s, v['thr'])
            unknown = [r for r in regs if not any(overlaps(r[1:], k[2:]) for k in known)]
        for cls, conf, x1, y1, x2, y2 in known:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('known', cls, conf, (off + x1, y1, off + x2, y2))
        for score, x1, y1, x2, y2 in unknown:
            if x1 > 2 and x2 < VIEW_W - 2:
                add('unknown', None, score, (off + x1, y1, off + x2, y2))
    torch.cuda.synchronize(); fps = len(OFFSETS) / (time.perf_counter() - t0)
    final = [t for t in tracks if t['hits'] >= MIN_HITS]
    matched = [any(same_defect(g[1:], t['box']) or overlaps(g[1:], t['box']) for t in final) for g in gt_roll]
    real = [any(overlaps(t['box'], g[1:]) for g in gt_roll) for t in final]
    metres = ROLL_LEN / PX_PER_M; rate = len(final) / metres * 100
    return {'labelled defects': len(gt_roll), 'defects counted': len(final), 'labelled defects found': int(sum(matched)),
            'counted but not labelled': int(len(final) - sum(real)),
            'known': int(sum(t['kind'] == 'known' for t in final)), 'unknown': int(sum(t['kind'] == 'unknown' for t in final)),
            'defects per 100 m': round(rate, 1), 'grade': 'A' if rate <= 5 else ('B' if rate <= 15 else 'C'),
            'frames per second': round(fps, 2)}

runs = [('original, PatchCore every frame', base, 1)]
if best is not base:
    runs += [(best['name'] + ', PatchCore every frame', best, 1), (best['name'] + ', PatchCore every 3rd frame', best, 3)]
else:
    runs += [('original, PatchCore every 3rd frame', base, 3)]
roll_rows = []
for name, v, every in runs:
    roll_rows.append({'setup': name, **run_roll(v, every)})
    print(roll_rows[-1])
roll_compare = pd.DataFrame(roll_rows)
roll_compare.to_csv(OUT / 'roll_compare.csv', index=False)
roll_compare.T

## 10. Save
Saves the two tables and, if a faster option was accepted, a new model file `patchcore_textile_fast.pt` with the smaller bank and its pixel threshold. Everything goes to the Drive folder `fabric_inspector_speed`.

In [ ]:
if best is not base:
    fast = {k: x for k, x in ck.items() if k != 'bank'}
    fast.update({'bank': best['bank'].b.cpu(), 'pix_thr': best['thr'], 'half': True, 'coreset_fraction': best['fraction']})
    torch.save(fast, OUT / 'patchcore_textile_fast.pt')
    print(f"patchcore_textile_fast.pt: {len(best['bank']):,} patches, pixel threshold {best['thr']:.2f}, "
          f"{(OUT / 'patchcore_textile_fast.pt').stat().st_size / 1e6:.0f} MB")

if DRIVE is not None:
    DEST = DRIVE / 'fabric_inspector_speed'; DEST.mkdir(exist_ok=True)
    for f in OUT.iterdir():
        shutil.copy(f, DEST / f.name)
    print("saved to", DEST, sorted(p.name for p in DEST.iterdir()))